# Ignite-3B S02 - Data build + tool smoke tests

Build all JSONL datasets + smoke test math_verify + code_exec + lean_tool.
Push HF dataset repo com todos os builds.

In [ ]:
!pip install -q -U 'datasets>=3.0.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'huggingface_hub' kaggle

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print('HF login OK')
except Exception as e:
    print(f'HF login skipped ({e}) - HF push at end will be skipped')

In [ ]:
import subprocess
for script in ['build_omni_math', 'build_livecodebench', 'build_bigcodebench', 'build_matharena', 'build_aime', 'build_putnam_lean']:
    print(f'=== {script} ===')
    subprocess.run(['python', '-m', f'data.ignite.{script}'], check=False)
subprocess.run(['ls', '-la', 'data/ignite/'])

In [ ]:
from eval.ignite.tools.math_verify import verify
assert verify('42', '42') == 1.0
assert verify('\\boxed{42}', '42') == 1.0
assert verify('\\boxed{99}', '42') == 0.0
print('math_verify OK')

In [ ]:
from eval.ignite.tools.code_exec import run_tests
p, t = run_tests('def add(a, b): return a + b', ['assert add(1, 2) == 3'])
assert (p, t) == (1, 1)
print('code_exec OK')

In [ ]:
import shutil
if shutil.which('lean') is None:
    print('lean not installed - skip smoke (add elan install if Kaggle image lacks it)')
else:
    from eval.ignite.tools.lean_tool import LeanDaemonPool
    pool = LeanDaemonPool(workers=1)
    try:
        r = pool.verify_proof('example : 1 + 1 = 2 := by rfl\n')
        print(f'lean smoke: proved={r.proved}')
    finally:
        pool.shutdown()

In [ ]:
import os
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    api = HfApi()
    repo = 'iterate-labs-ai/ignite-3b-data'
    api.create_repo(repo, repo_type='dataset', exist_ok=True, private=True)
    api.upload_folder(folder_path='data/ignite', repo_id=repo, repo_type='dataset', allow_patterns='*.jsonl')
    print(f'-> https://huggingface.co/datasets/{repo}')
else:
    print('HF_TOKEN not set - skip push. Data JSONLs available in /kaggle/working/caracal-1/data/ignite/')